# Imagen del dinosaurio (text-to-image)

Genera la imagen de Aborosaurus a partir del nombre y los rasgos que definió Ollama.
Se comparan dos modelos de tamaño similar con el mismo prompt:

| Modelo | Tamaño aprox. | Pasos |
|---|---|---|
| `amused/amused-512` | ~800 M parámetros | 12 |
| `stabilityai/sd-turbo` | ~1 300 M parámetros | 4 |

Ejecutar en Google Colab con GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q diffusers transformers accelerate safetensors

## Prompt

Parte del nombre y de la identidad de `ollama/identidad.json` (hábitat arbóreo, cuerpo esbelto, patas
largas, escamas verde oliva, herbívoro). La primera ronda mostró que el modelo necesita rasgos visuales
concretos, así que el prompt describe la pose y el estilo. El prompt negativo evita lo que contradice
la identidad: pico, dientes de carnívoro, cuerpo de cuadrúpedo pesado.

In [ ]:
import json, time, gc, torch
from PIL import Image

NOMBRE = "Aborosaurus"
PROMPT = (f"{NOMBRE}, a slender olive-green herbivorous dinosaur climbing a thick mossy tree trunk, "
          "long legs gripping the bark, small head, long tail, detailed scales, dense humid rainforest canopy, "
          "soft dappled sunlight, realistic nature documentary illustration")
NEGATIVO = "beak, bird, pterosaur, sharp teeth, quadruped, heavy body, blurry, deformed, extra limbs, text, watermark, cartoon"
SEMILLAS = list(range(8))
print(PROMPT)


def mosaico(imagenes, archivo, columnas=4):
    filas = (len(imagenes) + columnas - 1) // columnas
    m = Image.new("RGB", (512 * columnas, 512 * filas), "white")
    for i, img in enumerate(imagenes):
        m.paste(img.resize((512, 512)), (512 * (i % columnas), 512 * (i // columnas)))
    m.save(archivo)
    return m

## Modelo 1: aMUSEd 512
Mosaico: fila de arriba semillas 0-3, fila de abajo semillas 4-7.

In [ ]:
from diffusers import AmusedPipeline

pipe = AmusedPipeline.from_pretrained("amused/amused-512", variant="fp16", torch_dtype=torch.float16).to("cuda")
t = time.time()
amused = [pipe(PROMPT, negative_prompt=NEGATIVO, num_inference_steps=12, guidance_scale=10.0,
               generator=torch.Generator("cuda").manual_seed(s)).images[0] for s in SEMILLAS]
print(f"{(time.time() - t) / len(SEMILLAS):.1f} s por imagen")
del pipe; gc.collect(); torch.cuda.empty_cache()
mosaico(amused, "candidatas_amused.png")

## Modelo 2: SD-Turbo
Modelo destilado para generar en pocos pasos; no usa guía (`guidance_scale=0`), por eso ignora el prompt negativo.

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", variant="fp16", torch_dtype=torch.float16).to("cuda")
t = time.time()
turbo = [pipe(PROMPT, num_inference_steps=4, guidance_scale=0.0,
              generator=torch.Generator("cuda").manual_seed(s)).images[0] for s in SEMILLAS]
print(f"{(time.time() - t) / len(SEMILLAS):.1f} s por imagen")
mosaico(turbo, "candidatas_sd_turbo.png")

## Elección

Elige el modelo y la semilla de la imagen que mejor corresponde a la identidad
(esbelto, verde oliva, trepando en un árbol, bosque húmedo, sin rasgos de carnívoro).

In [ ]:
MODELO_ELEGIDO = "sd-turbo"   # "amused" o "sd-turbo"
SEMILLA_ELEGIDA = 0

imagenes = {"amused": amused, "sd-turbo": turbo}[MODELO_ELEGIDO]
imagenes[SEMILLAS.index(SEMILLA_ELEGIDA)].save(f"{NOMBRE.lower()}.png")
info = {
    "nombre": NOMBRE,
    "modelo": {"amused": "amused/amused-512", "sd-turbo": "stabilityai/sd-turbo"}[MODELO_ELEGIDO],
    "prompt": PROMPT,
    "negative_prompt": NEGATIVO if MODELO_ELEGIDO == "amused" else None,
    "pasos": 12 if MODELO_ELEGIDO == "amused" else 4,
    "guidance_scale": 10.0 if MODELO_ELEGIDO == "amused" else 0.0,
    "semilla": SEMILLA_ELEGIDA,
    "resolucion": "512x512",
    "gpu": torch.cuda.get_device_name(0),
}
with open("imagen_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

from google.colab import files
for archivo in [f"{NOMBRE.lower()}.png", "candidatas_amused.png", "candidatas_sd_turbo.png", "imagen_info.json"]:
    files.download(archivo)